# Aula Python · Consulta de apuntes con RAG

Abre este cuaderno en Google Colab y ejecuta las celdas en orden. Para la generación conviene elegir **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**, si está disponible. La primera consulta descarga el modelo. Los ejemplos de recuperación también funcionan en CPU.

## 1. Preparar las dependencias

Colab ya incluye PyTorch. La búsqueda usa la biblioteca estándar de Python.

In [ ]:
%pip -q install "transformers==4.51.3"

## 2. Leer los apuntes

Cada párrafo resume una operación de la documentación oficial y conserva su enlace. La división por tema permite citar el párrafo exacto. Puedes modificar un ejemplo aquí y volver a construir el índice.

In [ ]:
apuntes = [
  {
    "tema": "Listas",
    "texto": "Para agregar un elemento al final de una lista se usa append. Por ejemplo, notas.append(5) guarda la nueva nota después de las anteriores.",
    "fuente": "https://docs.python.org/es/3/tutorial/datastructures.html"
  },
  {
    "tema": "Listas",
    "texto": "Para ordenar una lista en el mismo sitio se usa sort. Para invertir el orden actual de sus elementos se usa reverse.",
    "fuente": "https://docs.python.org/es/3/tutorial/datastructures.html"
  },
  {
    "tema": "Pilas",
    "texto": "Una pila devuelve primero el último elemento que entró. En una lista, append agrega un elemento y pop() retira el último.",
    "fuente": "https://docs.python.org/es/3/tutorial/datastructures.html"
  },
  {
    "tema": "Colas",
    "texto": "Una cola atiende primero al que llegó primero. Con collections.deque, append agrega al final y popleft retira el elemento del frente.",
    "fuente": "https://docs.python.org/es/3/tutorial/datastructures.html"
  },
  {
    "tema": "Tuplas",
    "texto": "Una tupla mantiene sus posiciones y no permite reasignar sus elementos. Se puede desempaquetar una pareja con x, y = punto.",
    "fuente": "https://docs.python.org/es/3/tutorial/datastructures.html"
  },
  {
    "tema": "Conjuntos",
    "texto": "Un conjunto guarda valores sin duplicados. Para quitar elementos repetidos de una lista se puede convertir a set. El conjunto no conserva posiciones para acceder por índice.",
    "fuente": "https://docs.python.org/es/3/tutorial/datastructures.html"
  },
  {
    "tema": "Diccionarios",
    "texto": "Un diccionario asocia claves con valores. Se consulta un valor mediante su clave. El método items permite recorrer las parejas de clave y valor.",
    "fuente": "https://docs.python.org/es/3/tutorial/datastructures.html"
  }
]

## 3. Construir y consultar la base vectorial

Las palabras frecuentes en muchos fragmentos pesan menos. Guardamos los pesos y un vector normalizado por fragmento. La consulta SQL ordena los resultados por similitud coseno.

In [ ]:
"""Índice de vectores TF-IDF con búsqueda por coseno en SQLite."""
import json
import math
import re
import sqlite3
import unicodedata
from collections import Counter

VACIAS = set("a al con como cual de del el en es la las lo los metodo permite me mi para por que se sin su un una y".split())


def palabras(texto):
    plano = unicodedata.normalize("NFKD", texto.lower())
    plano = "".join(c for c in plano if not unicodedata.combining(c))
    return [p for p in re.findall(r"[a-z0-9_]+", plano) if p not in VACIAS]


def vector(texto, pesos):
    conteo = Counter(palabras(texto))
    valores = {p: n * pesos[p] for p, n in conteo.items() if p in pesos}
    norma = math.sqrt(sum(x * x for x in valores.values()))
    return {p: x / norma for p, x in valores.items()}


def indexar(ruta, apuntes):
    if not apuntes:
        raise ValueError("Agrega al menos un fragmento al corpus.")
    textos = [a["tema"] + " " + a["texto"] for a in apuntes]
    frecuencia = Counter(p for t in textos for p in set(palabras(t)))
    pesos = {p: 1 + math.log((1 + len(textos)) / (1 + n)) for p, n in frecuencia.items()}
    db = sqlite3.connect(ruta)
    try:
        # Reemplazamos todo en una transacción para que un apunte borrado desaparezca también del índice.
        with db:
            db.execute("CREATE TABLE IF NOT EXISTS terminos (palabra TEXT PRIMARY KEY, peso REAL)")
            db.execute("CREATE TABLE IF NOT EXISTS apuntes (id INTEGER PRIMARY KEY, tema TEXT, texto TEXT, fuente TEXT, vector TEXT)")
            db.execute("DELETE FROM terminos")
            db.execute("DELETE FROM apuntes")
            db.executemany("INSERT INTO terminos VALUES (?, ?)", pesos.items())
            for i, (a, t) in enumerate(zip(apuntes, textos), 1):
                db.execute("INSERT INTO apuntes VALUES (?, ?, ?, ?, ?)",
                           (i, a["tema"], a["texto"], a["fuente"], json.dumps(vector(t, pesos))))
    finally:
        db.close()


def recuperar(ruta, pregunta, limite=3, umbral=0.20):
    db = sqlite3.connect(ruta)
    db.row_factory = sqlite3.Row
    try:
        db.execute("BEGIN")
        consulta = vector(pregunta, dict(db.execute("SELECT palabra, peso FROM terminos")))
        if not consulta:
            return []
        # Como ambos vectores ya tienen longitud uno, su producto punto es el coseno.
        def coseno(guardado):
            documento = json.loads(guardado)
            return sum(v * documento.get(p, 0) for p, v in consulta.items())

        db.create_function("coseno", 1, coseno)
        filas = db.execute("""
            SELECT id, tema, texto, fuente, coseno(vector) AS puntaje FROM apuntes
            WHERE coseno(vector) >= ? ORDER BY puntaje DESC, id LIMIT ?
        """, (umbral, limite))
        return [dict(fila) for fila in filas]
    finally:
        db.close()

In [ ]:
ruta = "aula.sqlite3"
indexar(ruta, apuntes)
pregunta = "¿Cómo retiro al primero de una cola?"
evidencia = recuperar(ruta, pregunta)
for f in evidencia:
    print(f'[{f["id"]}] {f["puntaje"]:.3f} · {f["texto"]}\n{f["fuente"]}')

## 4. Evaluar la recuperación

Comparamos el primer resultado con el fragmento esperado. La última pregunta queda fuera del corpus. Esta prueba mide la selección de evidencia, no la calidad de redacción del modelo.

In [ ]:
def evaluar(ruta, casos):
    aciertos = 0
    for pregunta, esperado in casos:
        filas = recuperar(ruta, pregunta)
        primero = filas[0]["id"] if filas else None
        aciertos += primero == esperado
        print(f"{pregunta} | esperado={esperado}, recuperado={primero}")
    print(f"Aciertos: {aciertos}/{len(casos)}")
    return aciertos

casos = [["¿Cómo agrego una nota al final de una lista?",1],["¿Qué método permite ordenar una lista?",2],["¿Cómo funciona una pila?",3],["¿Cómo retiro al primero de una cola?",4],["¿Puedo reasignar elementos de una tupla?",5],["¿Cómo quito valores duplicados?",6],["¿Cómo recorro claves y valores de un diccionario?",7],["¿Cuánto cuesta viajar a Marte?",None]]
evaluar(ruta, casos)

## 5. Generar la respuesta

Qwen recibe la pregunta y los fragmentos recuperados. Comprueba que la cita respalde lo que dice la respuesta. Prueba después con otra pregunta cambiando la variable `pregunta` en la última celda.

In [ ]:
"""Generación local con un modelo instruct y referencias al corpus."""
import re
from functools import lru_cache

SIN_EVIDENCIA = "Los apuntes disponibles no alcanzan para responder esa pregunta."


@lru_cache(maxsize=1)
def cargar_modelo():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    nombre = "Qwen/Qwen2.5-1.5B-Instruct"
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    tipo = torch.float16 if dispositivo == "cuda" else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, torch_dtype=tipo)
    return tokenizer, modelo.to(dispositivo).eval()


def generar(pregunta, evidencia):
    if not evidencia:
        return SIN_EVIDENCIA

    import torch

    tokenizer, modelo = cargar_modelo()
    contexto = "\n".join(f'[{f["id"]}] {f["texto"]}' for f in evidencia)
    mensajes = [
        {"role": "system", "content":
         "Eres tutor de Python. Usa únicamente los apuntes como evidencia. "
         "Responde en español en una o dos frases y cita el número entre corchetes. "
         "Si falta información, responde exactamente: " + SIN_EVIDENCIA},
        {"role": "user", "content": f"Apuntes:\n{contexto}\n\nPregunta: {pregunta}"}
    ]
    prompt = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    with torch.inference_mode():
        salida = modelo.generate(**entrada, max_new_tokens=140, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
    # La salida también incluye la pregunta. Mostramos solo lo que escribió el modelo.
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    respuesta = tokenizer.decode(nuevos, skip_special_tokens=True).strip()
    if respuesta == SIN_EVIDENCIA:
        return respuesta
    citados = set(re.findall(r"\[(\d+)\]", respuesta))
    disponibles = {str(f["id"]) for f in evidencia}
    if not citados or not citados <= disponibles:
        return "El modelo no citó correctamente los apuntes. Revisa los fragmentos recuperados."
    return respuesta

In [ ]:
pregunta = "¿Cómo retiro al primero de una cola?"
evidencia = recuperar(ruta, pregunta)
print(generar(pregunta, evidencia))
for f in evidencia:
    print(f'[{f["id"]}] {f["tema"]}: {f["texto"]}\n{f["fuente"]}')